# Power Outage Analaysis

**Name(s)**: Ryo Ide

**Website Link**: (your website link)

In [60]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

As per the description on the course website, "this dataset has major power outage data in the continental U.S. from January 2000 to July 2016." I chose this dataset because I wanted to systematically discover the optimal resource allocation based on past trends, how many anomalies there are in a specific area, etc. Moreover, as a data science and physics student, Energy is an industry that I've had my eyes on for a while, and I wanted to get exposure through this project.

In [61]:
# TODO

df = pd.read_csv('data\outage.csv')
df

,variables,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL,CLIMATE.CATEGORY,...,POPPCT_URBAN,POPPCT_UC,POPDEN_URBAN,POPDEN_UC,POPDEN_RURAL,AREAPCT_URBAN,AREAPCT_UC,PCT_LAND,PCT_WATER_TOT,PCT_WATER_INLAND
0,Units,NaN,NaN,NaN,NaN,NaN,NaN,NaN,numeric,NaN,...,%,%,persons per square mile,persons per square mile,persons per square mile,%,%,%,%,%
1,NaN,1.0,2011.0,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59266587,8.407334131,5.478742983
2,NaN,2.0,2014.0,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59266587,8.407334131,5.478742983
3,NaN,3.0,2010.0,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59266587,8.407334131,5.478742983
4,NaN,4.0,2012.0,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,...,73.27,15.28,2279,1700.5,18.2,2.14,0.6,91.59266587,8.407334131,5.478742983
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530,NaN,1530.0,2011.0,12.0,North Dakota,ND,MRO,West North Central,-0.9,cold,...,59.9,19.9,2192.2,1868.2,3.9,0.27,0.1,97.59964921,2.401765255,2.401765255
1531,NaN,1531.0,2006.0,NaN,North Dakota,ND,MRO,West North Central,NaN,NaN,...,59.9,19.9,2192.2,1868.2,3.9,0.27,0.1,97.59964921,2.401765255,2.401765255
1532,NaN,1532.0,2009.0,8.0,South Dakota,SD,RFC,West North Central,0.5,warm,...,56.65,26.73,2038.3,1905.4,4.7,0.3,0.15,98.30774418,1.692255822,1.692255822
1533,NaN,1533.0,2009.0,8.0,South Dakota,SD,MRO,West North Central,0.5,warm,...,56.65,26.73,2038.3,1905.4,4.7,0.3,0.15,98.30774418,1.692255822,1.692255822


In [62]:
df.columns

Index(['variables', 'OBS', 'YEAR', 'MONTH', 'U.S._STATE', 'POSTAL.CODE',
       'NERC.REGION', 'CLIMATE.REGION', 'ANOMALY.LEVEL', 'CLIMATE.CATEGORY',
       'OUTAGE.START.DATE', 'OUTAGE.START.TIME', 'OUTAGE.RESTORATION.DATE',
       'OUTAGE.RESTORATION.TIME', 'CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL',
       'HURRICANE.NAMES', 'OUTAGE.DURATION', 'DEMAND.LOSS.MW',
       'CUSTOMERS.AFFECTED', 'RES.PRICE', 'COM.PRICE', 'IND.PRICE',
       'TOTAL.PRICE', 'RES.SALES', 'COM.SALES', 'IND.SALES', 'TOTAL.SALES',
       'RES.PERCEN', 'COM.PERCEN', 'IND.PERCEN', 'RES.CUSTOMERS',
       'COM.CUSTOMERS', 'IND.CUSTOMERS', 'TOTAL.CUSTOMERS', 'RES.CUST.PCT',
       'COM.CUST.PCT', 'IND.CUST.PCT', 'PC.REALGSP.STATE', 'PC.REALGSP.USA',
       'PC.REALGSP.REL', 'PC.REALGSP.CHANGE', 'UTIL.REALGSP', 'TOTAL.REALGSP',
       'UTIL.CONTRI', 'PI.UTIL.OFUSA', 'POPULATION', 'POPPCT_URBAN',
       'POPPCT_UC', 'POPDEN_URBAN', 'POPDEN_UC', 'POPDEN_RURAL',
       'AREAPCT_URBAN', 'AREAPCT_UC', 'PCT_LAND', 'PCT_WAT

## Step 2: Data Cleaning and Exploratory Data Analysis

Here, we clean the dataset by giving column names their respective units(if they have one). We also drop columns with NaNs, and convert numericals into floats/ints, because the default dataset stores them as strings.

In [63]:
# TODO
df = df.drop(columns=['variables'])
df.columns = df.columns + [f' ({col})' if pd.notna(col) else '' for col in df.loc[0, :]]
df = df.iloc[1:, :]
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except (ValueError, TypeError):
        pass  # leave text columns (state names, causes, etc.) alone
df

,OBS,YEAR,MONTH,U.S._STATE,POSTAL.CODE,NERC.REGION,CLIMATE.REGION,ANOMALY.LEVEL (numeric),CLIMATE.CATEGORY,"OUTAGE.START.DATE (Day of the week, Month Day, Year)",...,POPPCT_URBAN (%),POPPCT_UC (%),POPDEN_URBAN (persons per square mile),POPDEN_UC (persons per square mile),POPDEN_RURAL (persons per square mile),AREAPCT_URBAN (%),AREAPCT_UC (%),PCT_LAND (%),PCT_WATER_TOT (%),PCT_WATER_INLAND (%)
1,1.0,2011.0,7.0,Minnesota,MN,MRO,East North Central,-0.3,normal,2011�N7��1��,...,73.27,15.28,2279.0,1700.5,18.2,2.14,0.60,91.592666,8.407334,5.478743
2,2.0,2014.0,5.0,Minnesota,MN,MRO,East North Central,-0.1,normal,2014�N5��11��,...,73.27,15.28,2279.0,1700.5,18.2,2.14,0.60,91.592666,8.407334,5.478743
3,3.0,2010.0,10.0,Minnesota,MN,MRO,East North Central,-1.5,cold,2010�N10��26��,...,73.27,15.28,2279.0,1700.5,18.2,2.14,0.60,91.592666,8.407334,5.478743
4,4.0,2012.0,6.0,Minnesota,MN,MRO,East North Central,-0.1,normal,2012�N6��19��,...,73.27,15.28,2279.0,1700.5,18.2,2.14,0.60,91.592666,8.407334,5.478743
5,5.0,2015.0,7.0,Minnesota,MN,MRO,East North Central,1.2,warm,2015�N7��18��,...,73.27,15.28,2279.0,1700.5,18.2,2.14,0.60,91.592666,8.407334,5.478743
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1530,1530.0,2011.0,12.0,North Dakota,ND,MRO,West North Central,-0.9,cold,2011�N12��6��,...,59.90,19.90,2192.2,1868.2,3.9,0.27,0.10,97.599649,2.401765,2.401765
1531,1531.0,2006.0,NaN,North Dakota,ND,MRO,West North Central,NaN,NaN,NaN,...,59.90,19.90,2192.2,1868.2,3.9,0.27,0.10,97.599649,2.401765,2.401765
1532,1532.0,2009.0,8.0,South Dakota,SD,RFC,West North Central,0.5,warm,2009�N8��29��,...,56.65,26.73,2038.3,1905.4,4.7,0.30,0.15,98.307744,1.692256,1.692256
1533,1533.0,2009.0,8.0,South Dakota,SD,MRO,West North Central,0.5,warm,2009�N8��29��,...,56.65,26.73,2038.3,1905.4,4.7,0.30,0.15,98.307744,1.692256,1.692256


We define the function py``univariate_analysis(df, variable)`` as a universal function that, for a specific variable name/column in `df`, creates a box plot for that variable, as well as some aggregate statistics.

In [64]:
def univariate_analysis(df, variable):
    try:
        if isinstance(variable, str):
            series = df.loc[:, variable]
        elif isinstance(variable, int):
            series = df.iloc[:, variable]
    except KeyError:
        print(f"Invalid column name {variable} in the columns of dataframe")
        return None

    series.plot(kind='box').show()
    return series.describe()

Below is the univariate analysis for the variable `ANOMALY.LEVEL (numeric)`. According to the [data dictionary](https://www.sciencedirect.com/science/article/pii/S2352340918307182#t0005), this is "Oceanic El Niño/La Niña (ONI) index referring to the cold and warm episodes by season"

In [65]:
univariate_analysis(df, 'ANOMALY.LEVEL (numeric)')

count    1525.000000
mean       -0.096852
std         0.739957
min        -1.600000
25%        -0.500000
50%        -0.300000
75%         0.300000
max         2.300000
Name: ANOMALY.LEVEL (numeric), dtype: float64

Below is the univariate analysis for the variable `POPDEN_UC (persons per square mile)`. According to the [data dictionary](https://www.sciencedirect.com/science/article/pii/S2352340918307182#t0005), this is "Population density of the urban areas (persons per square mile)"

In [66]:
univariate_analysis(df, 'POPDEN_UC (persons per square mile)')

count    1524.000000
mean     1558.041142
std       313.149226
min       988.700000
25%      1307.800000
50%      1528.600000
75%      1732.200000
max      2220.700000
Name: POPDEN_UC (persons per square mile), dtype: float64

In [67]:
df.columns

Index(['OBS', 'YEAR', 'MONTH', 'U.S._STATE', 'POSTAL.CODE', 'NERC.REGION',
       'CLIMATE.REGION', 'ANOMALY.LEVEL (numeric)', 'CLIMATE.CATEGORY',
       'OUTAGE.START.DATE (Day of the week, Month Day, Year)',
       'OUTAGE.START.TIME (Hour:Minute:Second (AM / PM))',
       'OUTAGE.RESTORATION.DATE (Day of the week, Month Day, Year)',
       'OUTAGE.RESTORATION.TIME (Hour:Minute:Second (AM / PM))',
       'CAUSE.CATEGORY', 'CAUSE.CATEGORY.DETAIL', 'HURRICANE.NAMES',
       'OUTAGE.DURATION (mins)', 'DEMAND.LOSS.MW (Megawatt)',
       'CUSTOMERS.AFFECTED', 'RES.PRICE (cents / kilowatt-hour)',
       'COM.PRICE (cents / kilowatt-hour)',
       'IND.PRICE (cents / kilowatt-hour)',
       'TOTAL.PRICE (cents / kilowatt-hour)', 'RES.SALES (Megawatt-hour)',
       'COM.SALES (Megawatt-hour)', 'IND.SALES (Megawatt-hour)',
       'TOTAL.SALES (Megawatt-hour)', 'RES.PERCEN (%)', 'COM.PERCEN (%)',
       'IND.PERCEN (%)', 'RES.CUSTOMERS', 'COM.CUSTOMERS', 'IND.CUSTOMERS',
       'TOTAL.CUSTOMER

We define the function py``bivariate_analysis(df, variable_x, variable_y, x_type, y_type)`` as a universal function that, for a pair specific variable name/column in `df`, creates the most appropriate plot for the types of variables. That is, if both avriables are numerical, a scatter plot with be plotted. If one were categorical and the other numerical, a box plot will be plotted. Otherwise, a heatmap of counts will be shown.

In [68]:
def bivariate_analysis(df, variable_x, variable_y, x_type, y_type):
    assert any([isinstance(variable, str), isinstance(variable, int)] for variable in (variable_x, variable_y))
    assert all([var_type in ('categorical', 'numerical') for var_type in (x_type, y_type)])
        

    if x_type == 'numerical' and y_type == 'numerical':
        stats = df[variable_x].corr(df[variable_y])
        fig = px.scatter(df, x=variable_x, y=variable_y)
    elif x_type == 'categorical' and y_type == 'numerical':
        stats = df.groupby(variable_x)[variable_y].describe()
        fig = px.box(df, x=variable_x, y=variable_y)
    elif x_type == 'numerical' and y_type == 'categorical':
        stats = df.groupby(variable_y)[variable_x].describe()
        fig = px.box(df, x=variable_x, y=variable_y)
    else:
        stats = pd.crosstab(df[variable_x], df[variable_y])
        fig = px.imshow(stats, text_auto=True)          # heatmap of counts

    fig.show()
    return stats

In [69]:
bivariate_analysis(df, 'CLIMATE.REGION', 'POPDEN_UC (persons per square mile)', 'categorical', 'numerical')

,count,mean,std,min,25%,50%,75%,max
CLIMATE.REGION,,,,,,,,
Central,200.0,1558.591000,248.944169,1076.2,1351.7,1646.9,1740.1,1759.5
East North Central,138.0,1490.590580,153.197259,1390.4,1390.4,1390.4,1671.5,1834.5
Northeast,340.0,1411.700588,197.835965,1083.0,1291.8,1426.7,1528.6,1700.0
Northwest,132.0,1618.745455,218.771238,1487.9,1487.9,1487.9,1973.3,2004.7
South,229.0,1492.480786,158.763969,1280.1,1307.8,1539.9,1539.9,1983.0
Southeast,153.0,1140.642484,79.030219,988.7,1043.7,1179.2,1230.3,1230.3
Southwest,92.0,1634.171739,72.809404,1446.2,1611.2,1611.2,1669.0,1732.2
West,217.0,2096.635484,150.777164,1272.7,2124.1,2124.1,2124.1,2124.1
West North Central,17.0,1892.517647,196.605386,1732.4,1757.6,1757.6,1905.4,2220.7


For some aggregate analysis, we examine which state had the highest mean `DEMAND.LOSS.MW`, or "Amount of peak demand lost during an outage event (in Megawatt)".

In [72]:
demand_by_state = df.groupby('POSTAL.CODE')['DEMAND.LOSS.MW (Megawatt)'].mean().reset_index()

fig = px.choropleth(
    demand_by_state,
    locations='POSTAL.CODE',
    locationmode='USA-states',
    color='DEMAND.LOSS.MW (Megawatt)',
    scope='usa',
    color_continuous_scale='YlOrRd',
    title='Median Demand Loss per Outage by State (MW)',
)
fig.show()

## Step 3: Assessment of Missingness

In [ ]:
# TODO

## Step 4: Hypothesis Testing

Here, we examine whether severe-weather outages last longer in regions more expected to hurricanes. Namely, the Southeast (states including AL, FL, GA, NC, SC, VA) have higher Atlantic hurricane exposure, and we expect this region to have longer outages since power lines over wide areas are likely to take longer to repair. Moreover, answering this question will further inform which regions to allocate the most repair resources to.

**Null Hypothesis:** Severe-weather outages in the Southeast have teh same distribution of udration as severe-weather outages in the rest of the US. 

**Alternative Hyopthesis:** On average, severe-weather outages in the Southeast last longer than severe-weather outages in the rest of the US.

**Test Statistic:** The mean outage duration in the Southeast minus the mean outage duration in the rest of the US (minutes).

**Significance Level:** 0.05


In [73]:
# TODO

## Step 5: Framing a Prediction Problem

In [ ]:
# TODO

## Step 6: Baseline Model

In [ ]:
# TODO

## Step 7: Final Model

In [ ]:
# TODO

## Step 8: Fairness Analysis

In [ ]:
# TODO